## Sparse Autoencoders with MNIST

In this example an autoencoder is trained using L1 regularization on the hidden layer.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm.notebook import trange
import torch.nn.functional as F

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

### Prepare the Data

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

In [ ]:
train_x = train_dataset.data.reshape((60000, 28*28))
train_x = train_x.float() / 255
test_x = test_dataset.data.reshape((10000, 28 * 28))
test_x = test_x.float() / 255

In [ ]:
train_x = train_x.to(device)
train_dataset = TensorDataset(train_x)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, 
                                           shuffle=True, drop_last=True)

test_x = test_x.to(device)
test_dataset = TensorDataset(test_x)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, 
                                          shuffle=False, drop_last=True)

### Build the Autoencoder

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self):
        super(SparseAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28*28, 28*28),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(28*28, 28*28),
            nn.Linear(28*28, 28*28)
        )
        self.l1_lambda = 10e-5

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x
    
    def l1_regularizer(self):
        l1_loss = 0
        for param in self.encoder.parameters():
            l1_loss += torch.sum(torch.abs(param))
        return self.l1_lambda * l1_loss

In [ ]:
def train_sparse_autoencoder(model, train_loader, test_loader, num_epochs):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    history = {'train_loss': [], 'val_loss': []}

    for epoch in trange(num_epochs):
        model.train()
        train_loss = 0
        for [images] in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            mse_loss = criterion(outputs, images)
            l1_loss = model.l1_regularizer()
            loss = mse_loss + l1_loss
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)

        train_loss = train_loss / len(train_loader.dataset)
        history['train_loss'].append(train_loss)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for [images] in test_loader:
                outputs = model(images)
                mse_loss = criterion(outputs, images)
                l1_loss = model.l1_regularizer()
                loss = mse_loss + l1_loss
                val_loss += loss.item() * images.size(0)

        val_loss = val_loss / len(test_loader.dataset)
        history['val_loss'].append(val_loss)

        print(f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')

    return history

### Train the Model

In [ ]:
no_epochs = 200

In [ ]:
# Create and train the sparse autoencoder
sparse_auto = SparseAutoencoder().to(device)
sparse_history = train_sparse_autoencoder(sparse_auto, train_loader, test_loader, no_epochs)

### Display Images and Reconstructions

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
def reconstruct(model, test_loader):
    model.eval()
    reconstructions = []
    with torch.no_grad():
        for [images] in test_loader:
            outputs = model(images)
            reconstructions.append(outputs.cpu().numpy())

    return np.concatenate(reconstructions, axis=0)

In [ ]:
sparse_reconstruction = reconstruct(sparse_auto, test_loader)

In [ ]:
n = 10
plt.figure(figsize=(20, 4))

# Get reconstructions from the sparse autoencoder
sparse_reconstruction = reconstruct(sparse_auto, test_loader)

for i in range(n):
    # Display original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(test_loader.dataset[i][0].view(28, 28).cpu().detach().numpy(), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Display sparse reconstruction
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(sparse_reconstruction[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

plt.savefig('sparsereconstruct.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Extract the weights of the first layer (encoder)
hidden_weights = sparse_auto.encoder[0].weight.data.cpu().numpy()

In [ ]:
# Flatten the weights and take the absolute value
abs_hidden_weights = np.abs(hidden_weights.flatten())

In [ ]:
# Plot the histogram
num_bins = 20
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(abs_hidden_weights, num_bins, log=True, facecolor='gray', alpha=0.5)
ax.set_ylabel('Weight Density')
ax.set_xlabel('Magnitude')
fig.savefig('SparseWeightDensity.png', dpi=300, bbox_inches='tight')
plt.show()